In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib_map_utils import set_size
from matplotlib_map_utils.core.inset_map import (
    indicate_detail,
    indicate_extent,
    inset_map,
)

In [ ]:
set_size("small")

In [ ]:
zsj = gpd.read_parquet(
    "/data/uscuni-restricted/04_spatial_census/nadzsjd_education_2021.parquet"
)

In [ ]:
clusters = pd.read_csv(
    "/data/uscuni-restricted/geometries/cluster_assignment_v10.csv",
    dtype={"kod_nadzsj_d": str},
)
cluster_mapping = pd.read_parquet(
    "/data/uscuni-ulce/processed_data/clusters/cluster_mapping_v10.pq"
)
data = zsj.merge(clusters, left_on="nadzsjd", right_on="kod_nadzsj_d")
# variables = data.columns.drop(["geometry", "kod_nadzsj_d", "final_without_noise"])

mapped = data["final_without_noise"].map(cluster_mapping[3])
data["cluster"] = mapped

In [ ]:
# Define your custom color mapping
cluster_colors = {
    1: "#4069BC",
    2: "#7CBAE4",
    3: "#E69C63",
    4: "#eec1d5",
    5: "#E0665F",
    6: "#ECBF43",
    7: "#b2cd32",
    8: "#1F943E",
}

In [ ]:
# Map the colors to a new column
data["cluster_color"] = data["cluster"].map(cluster_colors)

In [ ]:
buildings = zsj = gpd.read_parquet("/data/uscuni-ulce/data_product/cz0.parquet")

In [ ]:
buildings = buildings.to_crs(5514)

In [ ]:
buildings

In [ ]:
buildings["cluster_color"] = buildings["level_3_label"].map(cluster_colors)

In [ ]:
prg = data.loc[data["naz_oblast"] == "Praha"]

In [ ]:
gpd.to_parquet("prg.parquet")

In [ ]:
data

In [ ]:
prg = data.loc[data["nazev_mco"] == "Praha 1"]

In [ ]:
# Create a buffer around the municipality
prague_buffer = prg.buffer(3000)

# get all buildings within the buffer
bp = gpd.sjoin(buildings, gpd.GeoDataFrame(geometry=prague_buffer), predicate="within")

In [ ]:
ax = bp.plot(figsize=(15, 10), color=bp["cluster_color"])

data.plot(facecolor="none", edgecolor="black", ax=ax)

minx, miny, maxx, maxy = prg.buffer(2000).total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)
ax.set_axis_off()

fig = ax.get_figure()
fig.savefig("more.png", dpi=300)

In [ ]:
ax = data.plot(figsize=(15, 10), color=data["cluster_color"])
ax.set_axis_off()

fig = ax.get_figure()

In [ ]:
ax = data.plot(color=data["cluster_color"], figsize=(15, 10), edgecolor="black")

# Plot buildings inside buffer
bp.plot(color="black", ax=ax)

# Set extent to the buffered area
minx, miny, maxx, maxy = prg.buffer(2000).total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

# Add inset map
iax = inset_map(ax, location="upper right", xticks=[], yticks=[], size=2.2)
data.plot(color=data["cluster_color"], ax=iax)
indicate_extent(pax=iax, bax=ax, pcrs=5514, bcrs=5514)